# S6 · Streamlit con memoria
## Miércoles 14 de octubre de 2026
Objetivo: explicar el rerun, guardar estado por sesión, usar keys y callbacks, enviar un form y conservar un historial.

No necesitamos S4, datasets ni modelos entrenados. `predict_toy()` está dado. Usa únicamente datos ficticios.

**Cómo usarlo:** ejecuta las celdas en orden, una a una (Shift+Enter). Cada paso explica, enseña el script y lo comprueba con una prueba automática. En los pasos con "Pruébalo tú", una celda arranca la app Streamlit real y la muestra dentro del notebook, para que pulses los botones.

Los mismos scripts están en la carpeta `scripts/`; fuera del notebook se abren con `python -m streamlit run <script>` desde esa carpeta.

In [2]:
from pathlib import Path
import sys
from streamlit.testing.v1 import AppTest
import streamlit
BASE = Path.cwd()
SCRIPTS = BASE / "scripts"
assert SCRIPTS.exists(), "Abre este notebook desde la carpeta S5_miercoles_streamlit"
sys.path.insert(0, str(SCRIPTS))   # para que las apps encuentren modelo.py
print("Streamlit:", streamlit.__version__)

Streamlit: 1.65.0


**Lanzar apps desde el notebook.** La celda siguiente define `lanzar()`: arranca `streamlit run` en segundo plano y muestra la app dentro del notebook. Así puedes pulsar los botones de verdad en cada paso. `parar()` las cierra.

In [ ]:
import subprocess, time, urllib.request
from IPython.display import IFrame, display

_procs = {}

def lanzar(script, puerto, alto=620):
    """Arranca una app Streamlit de scripts/ y la muestra aquí mismo (iframe).
    Si el iframe no aparece, abre el enlace http://localhost:<puerto> en otra pestaña."""
    parar(puerto)
    p = subprocess.Popen([sys.executable, "-m", "streamlit", "run", script,
                          "--server.headless", "true", "--server.port", str(puerto),
                          "--browser.gatherUsageStats", "false"],
                         cwd=SCRIPTS, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    _procs[puerto] = p
    for _ in range(40):
        try:
            if urllib.request.urlopen(f"http://localhost:{puerto}/_stcore/health", timeout=1).status == 200:
                break
        except Exception:
            time.sleep(0.5)
    print(f"App activa: http://localhost:{puerto}  ({script})")
    display(IFrame(f"http://localhost:{puerto}", width="100%", height=alto))

def parar(puerto=None):
    """Para una app (o todas si no das puerto)."""
    for pt in ([puerto] if puerto else list(_procs)):
        p = _procs.pop(pt, None)
        if p: p.terminate(); p.wait()

## 1. El contador roto (0-15 min)
El script empieza desde arriba en cada interacción normal. La variable local `count` vuelve a cero. Predice qué verás tras tres clics.

Demo real en terminal: `python -m streamlit run scripts/01_contador_roto.py`.


In [ ]:
broken = AppTest.from_file(str(SCRIPTS / "01_contador_roto.py")).run()
for _ in range(3):
    broken.button[0].click().run()
assert broken.metric[0].value == "1"
print("Tres clics, resultado:", broken.metric[0].value)


In [ ]:
lanzar("01_contador_roto.py", 8601, alto=300)

**Pruébalo tú:** pulsa Sumar varias veces. Siempre vuelve a 1.

## 2. Guardar memoria (15-30 min)
Lee `scripts/02_contador_estado.py`. Patrón: comprobar si la key existe → inicializar una vez → leer/escribir en `st.session_state`.

Si asignas `count = 0` al estado sin el guard, borras la memoria en cada rerun. Cada pestaña/conexión tiene su propia sesión; esto no es persistencia en disco.


In [ ]:
print((SCRIPTS / "02_contador_estado.py").read_text())
fixed = AppTest.from_file(str(SCRIPTS / "02_contador_estado.py")).run()
for _ in range(3):
    fixed.button[0].click().run()
assert fixed.metric[0].value == "3"
print("Tres clics, resultado:", fixed.metric[0].value)


In [ ]:
lanzar("02_contador_estado.py", 8602, alto=300)

**Pruébalo tú:** ahora la cuenta sube. Recarga la página del iframe: la sesión nueva empieza otra vez en 0.

## 3. Callback antes del rerun (30-45 min)
Lee el ejemplo. El callback cambia el estado antes de volver a dibujar el script. Pasa la función, no la llames: `on_click=increment`, no `on_click=increment()`.

No hace falta `st.rerun()` dentro de estos callbacks. Separa cambiar estado (callback) de mostrar estado (UI).


In [3]:
print((SCRIPTS / "03_contador_callback.py").read_text())
callback = AppTest.from_file(str(SCRIPTS / "03_contador_callback.py")).run()
callback.button[0].click().run()
callback.button[0].click().run()
assert callback.metric[0].value == "2"
print("Callback comprobado:", callback.metric[0].value)


2026-10-14 18:59:07.148 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


import streamlit as st
st.title("Contador con callback")
if "count" not in st.session_state:
    st.session_state.count = 0
def increment():
    st.session_state.count += 1
st.button("Sumar", on_click=increment)
st.metric("Cuenta", st.session_state.count)

Callback comprobado: 2


In [ ]:
lanzar("03_contador_callback.py", 8603, alto=300)

## 4. Formulario e historial (45-60 min)
Lee `scripts/04_app_ejemplo.py`. Todos los widgets tienen una key. El submit envía los valores juntos; dentro del form solo el botón submit puede tener callback.

El callback lee los valores recién enviados desde `st.session_state`, no desde args calculados al dibujar el form. Guarda una fila únicamente al enviar. El historial se dibuja fuera del form y fuera del callback.

Abre la app: `python -m streamlit run scripts/04_app_ejemplo.py`. Cambia un input sin enviar: no debería añadirse una fila. Envía tres veces y activa Mostrar ayuda: el historial debe seguir en tres.


In [4]:
print((SCRIPTS / "04_app_ejemplo.py").read_text())
from modelo import predict_toy
print(predict_toy(6, 80.0, "Mensual", 2))
app = AppTest.from_file(str(SCRIPTS / "04_app_ejemplo.py")).run()
assert not app.exception
assert len(app.session_state.history) == 0
submit = next(b for b in app.button if b.label == "Predecir y guardar")
for _ in range(3):
    submit.click().run()
assert len(app.session_state.history) == 3
app.checkbox[0].check().run()
assert len(app.session_state.history) == 3
assert app.metric[0].value == "3"
print("Tres submits y un rerun de ayuda: historial =", len(app.session_state.history))


2026-10-14 18:59:12.379 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


import streamlit as st
from modelo import predict_toy

st.set_page_config(page_title="Churn con memoria", layout="centered")
st.title("Churn con memoria")
st.caption("Regla de juguete. No usar para tomar decisiones sobre clientes.")

# A: iniciar una sola vez, no en cada rerun.
if "history" not in st.session_state:
    st.session_state.history = []

def submit_prediction():
    # Leer los valores nuevos desde las keys, no mediante args del callback.
    inputs = {name: st.session_state[name] for name in
              ("tenure_months", "monthly_spend_eur", "contract", "support_calls")}
    result = predict_toy(**inputs)
    # B: una fila por submit. No predecir en cada rerun.
    st.session_state.history.append({**inputs, **result})

def clear_history():
    st.session_state.history = []

with st.form("churn_form"):
    st.slider("Antigüedad (meses)", 0, 72, 6, key="tenure_months")
    st.number_input("Gasto mensual (EUR)", 0.0, 200.0, 80.0,
                    step=5.0, key="monthly_sp

In [ ]:
lanzar("04_app_ejemplo.py", 8604, alto=900)

**Pruébalo tú:** cambia inputs sin enviar (no hay fila), envía 3 veces, marca *Mostrar ayuda* (sigue en 3), pulsa *Vaciar historial*.

## 5. Tu práctica (60 min)
Abre `scripts/05_practica_starter.py` y completa A-C:

- A: inicializa `history` con un guard.
- B: añade `{**inputs, **result}` en el callback del submit.
- C: muestra `st.metric` y `st.dataframe` a partir del estado.

Comandos desde la carpeta S5_miercoles_streamlit:
```bash
python -m streamlit run scripts/05_practica_starter.py
```
La plantilla arranca sin errores, pero aún no guarda historial. No necesitas modificar el predictor.

**Comprueba:** 0 filas al abrir; 3 tras tres submits; activar ayuda no crea fila; Vaciar historial vuelve a 0. Entrega el código y capturas de esas comprobaciones según el procedimiento de clase. No diseñes tests, arquitectura ML ni bundles.

**Pregunta:** si un widget deja de dibujarse, ¿su key sigue siendo una buena memoria permanente? No: copia lo enviado a una key de datos independiente. Lo usarás el viernes.


In [ ]:
# Abre la plantilla incompleta. Edita scripts/05_practica_starter.py en tu editor y recarga el iframe (R).
lanzar("05_practica_starter.py", 8605, alto=900)

## 6. Qué recordar
- Estado por sesión: `st.session_state`. Caché: reutilizar cálculos/recursos, compartidos por defecto entre sesiones; no usar la caché como historial privado. La API también permite scope de sesión en versiones actuales; queda fuera de esta práctica.
- Form: cambios enviados en bloque. Callback: actualiza estado antes del rerun.
- Página ≠ paso. Viernes: varias páginas y un pequeño flujo por pasos dentro de una de ellas.
- Refresh, desconexión o reinicio del servidor pueden perder el estado. Para guardarlo de forma permanente hace falta almacenamiento, fuera de esta semana.

## Fuentes oficiales
https://docs.streamlit.io/develop/concepts/architecture/session-state

https://docs.streamlit.io/develop/concepts/architecture/forms

https://docs.streamlit.io/develop/concepts/multipage-apps/widgets

https://docs.streamlit.io/develop/concepts/multipage-apps/page-and-navigation

https://docs.streamlit.io/develop/concepts/architecture/caching


In [ ]:
# Al terminar la clase: cierra las apps lanzadas desde el notebook.
# (También se cierran solas al apagar el kernel.)
# Quita el "#" de la línea siguiente y ejecuta la celda:
# parar()